In [ ]:
import pandas as pd
import numpy as np

from janome.tokenizer import Tokenizer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

1.embeddingするテキストを準備

In [ ]:
texts = [
    "太陽光発電量予測",
    "電力需要予測",
    "犬の写真",
    "白い犬",
    "銀行マーケティング",
    "画像分類",
    "炎上記事",
    "dockerコンテナ"
]

2.TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer()
vectorized = vectorizer.fit_transform(texts)

In [ ]:
vectorized.shape

In [ ]:
# 類似度求めてみる

similarity_matrix = cosine_similarity(vectorized)

In [ ]:
df_similarity_matrix_vec = pd.DataFrame(
    similarity_matrix,
    index=texts,
    columns=texts
)

In [ ]:
# コサイン類似度の確認
df_similarity_matrix_vec

# →自分自身以外の類似度が０になった。

3.sentence_transformers

3.1単語レベルで動作確認

In [ ]:
#本来は英語特化モデル.ただし、日本語でも一部動作することが技術レポートで確認済み
model = SentenceTransformer("all-MiniLM-L6-v2") 

In [ ]:
# 意味空間に変換
embeddings_text = model.encode(texts)

In [ ]:
embeddings_text.shape

In [ ]:
# 文字間のコサイン類似度を計算
similarity_matrix_text = cosine_similarity(embeddings_text)


In [ ]:
df_similarity_matrix_text = pd.DataFrame(
    similarity_matrix,
    index=texts,
    columns=texts
)

df_similarity_matrix_text

3.2短文にして動作確認

In [ ]:
sentences = [
    "太陽光発電量を予測する回帰モデル",
    "電力需要を予測する回帰モデル",
    "犬の写真",
    "白い犬",
    "銀行におけるマーケティング施策の成否を予測する分類モデル",
    "気象に関連する画像を分類するモデル",
    "炎上したネット記事",
    "dockerでコンテナを作成する"
]

In [ ]:
embeddings_sentences = model.encode(sentences)

In [ ]:
embeddings_sentences.shape

In [ ]:
similarity_matrix_sentences = cosine_similarity(embeddings_sentences)


In [ ]:
df_similarity_matrix_sentences = pd.DataFrame(
    similarity_matrix_sentences,
    index=sentences,
    columns=sentences
)

df_similarity_matrix_sentences

4.embbedingsを用いた簡易検索

In [ ]:
def query_search(query, top_k):
    
    query_embedding = model.encode(query)
    cos_scores = cosine_similarity(query_embedding.reshape(1, -1), embeddings_sentences)

    # 類似度トップのみを表示
    # top_kの引数を外す
    # best_match_idx = cos_scores.argmax().item()
    # print("検索結果：", sentences[best_match_idx])

    # スコアの大きい順にしてtop_kのインデックスを取り出す
    top_indices = np.argsort(cos_scores[0])[::-1][:top_k]

    for i, j in enumerate(top_indices, start=1):
        print(f"検索結果{i}位(類似度:{cos_scores[0][j]:.2f})：", sentences[j])




In [ ]:
query = "犬は偉大！！"

query_search(query=query, top_k=3)